# 수위(level)를 넣은 ARX 온도변화 모델

기존 ARX(`ARX_Temp_Model.py`)에 **수위**를 추가해서 얼마나 좋아지는지, 그리고
**고장 감지 능력에는 어떤 영향이 있는지**를 같이 본다.

## 넣는 값들

| 특징 | 뜻 | 왜 넣나 |
|---|---|---|
| `T` | 현재 온도 | 뉴턴 냉각 — 뜨거울수록 더 내려간다 |
| `heat_fut` / `heat_past` | 앞으로/지난 60초 히터 ON 비율 | 가열 입력. cool 은 heat 과 완전 상보라 제외 |
| `exch_fut` / `exch_past` | exchanger ON 비율 | |
| `feed_fut` / `feed_past` | feeding ON 비율 | **명령 신호** |
| `level` | 현재 수위 | **열용량.** 액체가 많을수록 같은 열을 줘도 온도가 덜 움직인다 |
| `dlevel_fut` / `dlevel_past` | 60초 동안의 수위 변화 | **실제로 들어온 양.** feeding 이진 신호보다 해상도가 높다 |

세 가지 모델을 비교한다.

- **M1** = 기존 모델 (T, heat, exch, feed)
- **M2** = M1 + `level` (열용량만 추가)
- **M3** = M2 + `dlevel` (실제 유입량까지 추가)

## ⚠️ 먼저 알아야 할 함정 — 정확도와 고장 감지는 서로 상충한다

`dlevel` 은 **feeding 릴레이의 결과물**이다. 그래서 이걸 모델에 넣으면:

- **정확도는 올라간다** (실제 들어온 양을 알려주니까)
- **그런데 feeding 릴레이 고장은 못 잡게 된다**

이유: feeding 릴레이가 죽으면 `feed=1`(명령은 나감) 인데 액체는 안 들어와서
`dlevel=0` 이고 온도도 안 오른다. `dlevel` 을 보는 모델은 "수위가 안 올랐으니
온도도 안 오르겠네" 라고 **정확히 예측해버린다.** 잔차가 작으니 알람이 안 울린다.

**고장 감지 모델은 "명령"으로 굴러가야지 "측정된 결과"로 굴러가면 안 된다.**
아래에서 이걸 숫자로 확인한다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG

import matplotlib
import matplotlib.pyplot as plt
# ↑ 위 import 가 백엔드를 Agg 로 바꾸므로 inline 은 반드시 그 뒤에
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"

GRID_SEC = 5
HORIZON_SEC = 60
STEPS = HORIZON_SEC // GRID_SEC
EXCLUDE_HOURS = (0, 7)
RUNNING_WINDOW_MIN = 30
TEST_FRACTION = 0.3

M1 = ["T", "heat_fut", "heat_past", "exch_fut", "exch_past", "feed_fut", "feed_past"]
M2 = M1 + ["level"]
M3 = M2 + ["dlevel_fut", "dlevel_past"]
MODELS = {"M1_기존": M1, "M2_+수위": M2, "M3_+수위변화": M3}

## 1. 특징 만들기

특징은 **끊기지 않은 전체 시계열 위에서 먼저** 만들고, 제외 구간은 그 다음에 버린다.
(00~07시를 먼저 지우면 롤링 윈도우가 구멍을 건너뛰어 붙어버린다)

제외 대상: 00~07시, 알람(DIAGNOSTIC) 구간, **라인 정지 구간**
(wagon 카운터가 30분간 안 움직이면 정지로 판단 — 9/13 은 하루 종일 정지였다)

In [ ]:
_CACHE = {}


def build(tank_key):
    if tank_key in _CACHE:
        return _CACHE[tank_key]

    cfg = TANKS[tank_key]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True).sort_index()

    num = pd.DataFrame({
        "T": raw[cfg["temp"]], "level": raw[cfg["level"]],
        "heat": raw[cfg["heat"]], "cool": raw[cfg["cool"]],
        "exch": raw[cfg["exchanger"]], "feed": raw[cfg["feeding"]],
        "wagon": raw[WAGON_TAG],
        "diag": (raw["source"] == "DIAGNOSTIC").astype(float),
    })
    g = num.resample(f"{GRID_SEC}s").ffill().dropna()

    out = pd.DataFrame(index=g.index)
    out["T"] = g["T"]
    out["level"] = g["level"]
    out["y"] = g["T"].shift(-STEPS) - g["T"]

    for nm in ["heat", "exch", "feed"]:
        out[f"{nm}_fut"] = g[nm].rolling(STEPS).mean().shift(-(STEPS - 1))
        out[f"{nm}_past"] = g[nm].rolling(STEPS).mean().shift(1)

    out["dlevel_fut"] = g["level"].shift(-STEPS) - g["level"]
    out["dlevel_past"] = g["level"] - g["level"].shift(STEPS)

    moved = (g["wagon"].diff().abs() > 0).astype(float)
    running = moved.rolling(int(RUNNING_WINDOW_MIN * 60 / GRID_SEC), min_periods=1).max()
    hour = g.index.hour
    ok = (((hour < EXCLUDE_HOURS[0]) | (hour >= EXCLUDE_HOURS[1])) & (g["diag"] == 0)).astype(float) * running
    out["ok"] = ok.rolling(2 * STEPS + 1, center=True).min()

    out = out.dropna()
    out = out[out["ok"] == 1].drop(columns="ok")
    _CACHE[tank_key] = out
    return out


for tk in TANKS:
    d = build(tk)
    print(f"[{tk}] 유효표본 {len(d):,}  |  {d.index.min()} ~ {d.index.max()}")

## 2. 학습 / 평가 (시간순 분할)

In [ ]:
def ols(X, y):
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / max(len(y) - X.shape[1], 1)
    return beta, np.sqrt(np.diag(sigma2 * XtX_inv))


def fit_eval(df, features):
    n = len(df)
    sp = int(n * (1 - TEST_FRACTION))
    tr, te = df.iloc[: sp - STEPS], df.iloc[sp:]    # 경계에 purge

    Xtr = np.column_stack([np.ones(len(tr))] + [tr[f].values for f in features])
    Xte = np.column_stack([np.ones(len(te))] + [te[f].values for f in features])
    ytr, yte = tr["y"].values, te["y"].values

    beta, se = ols(Xtr, ytr)
    pred = Xte @ beta
    return {
        "beta": beta, "se": se, "names": ["const"] + features,
        "rmse": float(np.sqrt(np.mean((yte - pred) ** 2))),
        "r2": float(1 - np.sum((yte - pred) ** 2) / np.sum((yte - yte.mean()) ** 2)),
        "test": te, "pred": pred, "actual": yte,
    }

## 3. 세 모델 비교

In [ ]:
rows = []
FIT = {}
for tk in TANKS:
    d = build(tk)
    FIT[tk] = {}
    r = {"탱크": tk}
    for name, feats in MODELS.items():
        res = fit_eval(d, feats)
        FIT[tk][name] = res
        r[f"{name}_R2"] = round(res["r2"], 3)
        r[f"{name}_RMSE"] = round(res["rmse"], 3)
    rows.append(r)

cmp = pd.DataFrame(rows)
cmp

In [ ]:
# RMSE 개선율 (M1 대비)
imp = pd.DataFrame({
    "탱크": cmp["탱크"],
    "M1_RMSE": cmp["M1_기존_RMSE"],
    "M2_개선%": ((cmp["M1_기존_RMSE"] - cmp["M2_+수위_RMSE"]) / cmp["M1_기존_RMSE"] * 100).round(1),
    "M3_개선%": ((cmp["M1_기존_RMSE"] - cmp["M3_+수위변화_RMSE"]) / cmp["M1_기존_RMSE"] * 100).round(1),
})
imp

## 4. 계수 — heat 과 feed 가 어떻게 되는지

In [ ]:
def coef_table(model_name, keys=("T", "level", "heat_fut", "exch_fut", "feed_fut", "dlevel_fut")):
    rows = []
    for tk in TANKS:
        res = FIT[tk][model_name]
        r = {"탱크": tk}
        for k in keys:
            if k in res["names"]:
                i = res["names"].index(k)
                b, s = res["beta"][i], res["se"][i]
                r[k] = round(float(b), 3)
                r[k + "_t"] = round(float(b / s), 1) if s > 0 else None
            else:
                r[k] = None
                r[k + "_t"] = None
        rows.append(r)
    return pd.DataFrame(rows)

print("=== M1 (기존) ===")
display(coef_table("M1_기존"))
print("=== M3 (수위 변화까지) ===")
display(coef_table("M3_+수위변화"))

## 5. 고장 감지 능력 — 여기가 핵심

각 액추에이터가 죽었다고 가정했을 때 잔차가 얼마나 튀는지 본다.

**검출비 = (그 액추에이터의 온도 기여량) ÷ (모델 잔차 표준편차)**

1 이상이면 한 번만 봐도 티가 난다. **M1 과 M3 를 비교하면, 수위를 넣었을 때
feeding 고장 감지력이 어떻게 되는지 바로 보인다.**

In [ ]:
def detectability(model_name):
    rows = []
    for tk in TANKS:
        res = FIT[tk][model_name]
        te, sigma = res["test"], res["rmse"]
        for act, feat in [("heat", "heat_fut"), ("exchanger", "exch_fut"), ("feeding", "feed_fut")]:
            i = res["names"].index(feat)
            x = te[feat].values
            m = x > 0.01
            if m.sum() < 30:
                rows.append({"탱크": tk, "액추에이터": act, "검출비": None})
                continue
            eff = abs(res["beta"][i] * x[m].mean())
            rows.append({"탱크": tk, "액추에이터": act, "검출비": round(eff / sigma, 3)})
    return pd.DataFrame(rows).pivot(index="탱크", columns="액추에이터", values="검출비")

d1 = detectability("M1_기존")
d3 = detectability("M3_+수위변화")
side = pd.concat({"M1_기존": d1, "M3_+수위변화": d3}, axis=1)
side

## 6. 그래서 feeding 진단은 온도가 아니라 **수위로** 해야 한다

온도를 거치지 않고 바로 확인한다: **feeding 이 켜졌을 때 수위가 실제로 올랐는가?**

릴레이가 죽으면 명령은 나가는데 수위가 안 오른다 — 이건 열 지연도, 제어 루프
역인과도, 다른 액추에이터와의 얽힘도 없는 **직접 관측**이다.

In [ ]:
rows = []
for tk in TANKS:
    d = build(tk)
    on = d["feed_fut"] > 0.01
    if on.sum() < 30:
        rows.append({"탱크": tk, "feeding구간": int(on.sum()), "판정": "표본부족"})
        continue
    up_on = d.loc[on, "dlevel_fut"]
    up_off = d.loc[~on, "dlevel_fut"]
    pooled = np.sqrt((up_on.var() * (len(up_on) - 1) + up_off.var() * (len(up_off) - 1))
                     / (len(up_on) + len(up_off) - 2))
    dcoh = (up_on.mean() - up_off.mean()) / pooled if pooled > 0 else np.nan
    rows.append({
        "탱크": tk, "feeding구간": int(on.sum()),
        "수위변화_ON": round(up_on.mean(), 3), "수위변화_OFF": round(up_off.mean(), 3),
        "Cohen_d": round(dcoh, 2),
        "판정": "검출가능" if abs(dcoh) > 0.8 else ("약함" if abs(dcoh) > 0.3 else "불가"),
    })
pd.DataFrame(rows)

## 7. 예측 결과 눈으로 보기

In [ ]:
PLOT_TANK = "I3"
PLOT_MODEL = "M3_+수위변화"

res = FIT[PLOT_TANK][PLOT_MODEL]
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(res["test"].index, res["actual"], color="#444444", linewidth=0.8, label="실제 ΔT")
ax.plot(res["test"].index, res["pred"], color="#d62728", linewidth=0.8, alpha=0.85, label="예측 ΔT")
ax.set_title(f"{PLOT_TANK} — {PLOT_MODEL}   R²={res['r2']:.3f}  RMSE={res['rmse']:.2f}")
ax.set_ylabel("ΔT (60초)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()